# Clean public-domain poems

Reads `data/raw/poems.json` and writes `data/processed/poems.csv`.

Drops authors who died in 1956 or later (still in copyright in Australia), and English translations whose translator is not named. Edward Powys Mathers is kept: his translations are labeled, and he died in 1939.

Removes editors' material: footnotes, page numbers, ballad notes ("The Text is from...") and prose lines over 200 characters. Entries that are mostly prose are dropped, and so are epics and other book-length works over 1,700 words (about 3,000 tokens). This runs before scoring, so notes do not sway the genre.

Then [bhadresh-savani/distilbert-base-uncased-emotion](https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion) scores each poem. `Genre` is the highest of the six emotions. The other columns are those probabilities, from 0 to 1. Poems longer than 512 tokens are scored on the opening.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import torch
from transformers import pipeline

def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "poems.json").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/poems.json")


root = repo_root()
raw_path = root / "data" / "raw" / "poems.json"
out_path = root / "data" / "processed" / "poems.csv"

EMOTION_LABELS = ["sadness", "joy", "love", "anger", "fear", "surprise"]

In [ ]:
# Died in 1956 or later, or still alive.
still_in_copyright = {
    "A. R. Ammons",
    "Alan Alexander Milne",
    "Aldous Leonard Huxley",
    "Alfred Noyes",
    "Anna Akhmatova",
    "Bertolt Brecht",
    "Clark Ashton Smith",
    "Conrad Potter Aiken",
    "Edmund Charles Blunden",
    "Elizabeth Jennings",
    "Fay Inchfawn",
    "Frank James Prewett",
    "John Collings Squire, Sir",
    "John Leroy Atwell aka Roy Atwell",
    "John Masefield",
    "Martin Armstrong",
    "Muriel Stuart",
    "Nizar Qabbani",
    "Paul Bewsher",
    "Paul Cameron Brown",
    "Peter Courtney Quennell, Sir",
    "Ralph Hodgson",
    "Richard Arthur Warren Hughes",
    "Robert Lee Frost",
    "Robert Lowell",
    "Robert von Ranke Graves",
    "Robert William Service",
    "Siegfried Loraine Sassoon",
    "Thomas Moult",
    "Thomas R. Ybarra",
    "Victoria Mary Sackville-West",
    "Wallace Irwin",
    "Walter De La Mare",
    "Wilfrid Wilson Gibson",
    "Yehuda Amichai",
}

# Original poet did not write these lines in English, and the translator is unnamed.
unnamed_translations = {
    "Abraham a Sancta-Clara",
    "Adam Bernard Mickiewicz",
    "Andrejs Pumpurs",
    "Ant'nio Corr'a",
    "Aristophanes",
    "Aurelius Clemens Prudentius",
    "Bj'rnstjerne Martinius Bj'rnson",
    "Charles Baudelaire",
    "Dante Alighieri",
    "Fernando Ant'nio Nogueira Pessoa",
    "Francesco Petrarca (Petrarch)",
    "Friedrich Schiller",
    "Giacomo Leopardi",
    "Heinrich Hoffmann",
    "Ivan Sergeyevich Turgenev",
    "Jan Kochanowski",
    "Jean de La Fontaine",
    "Johann Wolfgang von Goethe",
    "Michelangelo di Lodovico Buonarroti Simoni",
    "Sophocles",
    "Tomas de Iriarte y Oropesa",
    "Victor-Marie Hugo",
}

In [ ]:
poems = pd.read_json(raw_path)
print("loaded", len(poems))

poems = poems[~poems["Author"].isin(still_in_copyright)]
print("after copyright drop", len(poems))

poems = poems[~poems["Author"].isin(unnamed_translations)]
print("after translation drop", len(poems))

In [ ]:
def clean_poem(value: str) -> str:
    text = value.replace("\xa0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)


def clean_title(value: str) -> str:
    return " ".join(clean_poem(value).split())


poems["Title"] = poems["Title"].map(clean_title)
poems["Poem"] = poems["text"].map(clean_poem)
poems = poems[(poems["Title"] != "") & (poems["Poem"] != "")]
print("after empty drop", len(poems))

poems = poems.drop_duplicates(subset=["Poem"], keep="first")
print("after duplicate drop", len(poems))

poems = poems[["Title", "Poem"]].reset_index(drop=True)

In [ ]:
# Editors' notes and prose. 99% of verse lines are under 72 characters and Whitman's
# run to about 200, so longer lines are notes, prefaces or stage directions.
PROSE_LINE_CHARS = 200
# Case-sensitive so verse such as "The story of their westward march." is kept.
NOTE_LINE = re.compile(r"^The (Text|Story)( is| was| of this|\.--)")
# [Footnote 3: ...], [* Impeach, hinder.] and page or line numbers like [460].
# Other brackets are often stage directions ("[Scene closes.]") and stay.
BRACKET_NOTE = re.compile(r"[ \t]*(?:\[(?:Footnote\b|\*)[^\]]*\]|\[\d+\])")
# Epics, book-length works and verse plays. About 3,000 tokens at 1.74 tokens per word;
# training only ever sees their openings, which read like the start of a book.
MAX_WORDS = 1700


def is_prose(poem: str) -> bool:
    # Judged on line lengths, not the title: titles like "Poems and Prose Remains" hold verse.
    long_chars = sum(len(line) for line in poem.split("\n") if len(line) > PROSE_LINE_CHARS)
    return long_chars > 0.5 * len(poem)


def remove_notes(poem: str) -> str:
    poem = BRACKET_NOTE.sub("", poem)
    lines = [
        line
        for line in poem.split("\n")
        if len(line) <= PROSE_LINE_CHARS and not NOTE_LINE.match(line)
    ]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


poems = poems[~poems["Poem"].map(is_prose)]
print("after prose drop", len(poems))

poems = poems.assign(Poem=poems["Poem"].map(remove_notes))
line_counts = poems["Poem"].map(lambda p: sum(1 for line in p.split("\n") if line.strip()))
poems = poems[line_counts >= 2]
print("after note removal and short drop", len(poems))

poems = poems[poems["Poem"].str.split().str.len() <= MAX_WORDS].reset_index(drop=True)
print("after epic drop", len(poems))

In [ ]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("emotion device", device)

classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    top_k=None,
    truncation=True,
    device=device,
)

score_columns = {label: [] for label in EMOTION_LABELS}
genres = []
texts = poems["Poem"].tolist()
batch_size = 32

for start in range(0, len(texts), batch_size):
    batch = texts[start:start + batch_size]
    predictions = classifier(batch, batch_size=batch_size, truncation=True, max_length=512)
    for prediction in predictions:
        by_label = {item["label"]: float(item["score"]) for item in prediction}
        winner = max(EMOTION_LABELS, key=lambda label: by_label[label])
        genres.append(winner.capitalize())
        for label in EMOTION_LABELS:
            score_columns[label].append(round(by_label[label], 4))
    done = min(start + batch_size, len(texts))
    if done % 3200 == 0 or done == len(texts):
        print("scored", done, "/", len(texts))

poems["Genre"] = genres
for label in EMOTION_LABELS:
    poems[label.capitalize()] = score_columns[label]

poems = poems[
    ["Title", "Poem", "Genre", "Sadness", "Joy", "Love", "Anger", "Fear", "Surprise"]
]
poems.to_csv(out_path, index=False)
print("wrote", out_path.resolve(), poems.shape)
print(poems["Genre"].value_counts().to_string())